In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet("./output/natality_2021_numeric.parquet")

In [ ]:
df = df.loc[df["previous_cesarean"] == 1]

df.describe()

In [ ]:
flag_column_names = df.filter(regex="(.*imputed.*|.*flag.*)$", axis=1).columns

flag_column_names_list = flag_column_names.to_list()

unrelated_features = [
    "birth_year",
    "reported_mothers_age_used",
    "five_minute_apgar_score",
    "five_minute_apgar_recode",
    "ten_minute_apgar_score",
    "ten_minute_apgar_score_recode",
    "plurality_recode",
    "set_order_recode",
    "last_normal_menses_month",
    "last_normal_menses_year",
    "obstetric_estimate_edited",
    "assisted_ventilation_immediate",
    "assisted_ventilation_after_6_hours",
    "admission_to_nicu",
    "surfactant",
    "antibiotics_for_newborn",
    "seizures",
    "no_abnormal_conditions_reported",
    "infant_transferred",
    "infant_living_at_time_of_report",
    "infant_breastfed_at_discharge",
    "month_prenatal_care_started",
    "no_maternal_comorbidity_reported",
    "perineal_laceration",
    "ruptured_uterus",
    "total_birth_order_recode",
    "unplanned_hysterectomy",
    "maternal_transfusion",
    "admit_to_intensive_care",
    "live_birth_order_recode",
    "2021",
]

df_filtered = df.drop(flag_column_names_list, axis="columns").drop(
    unrelated_features, axis="columns"
)

In [ ]:
df_filtered["successful_vbac"] = df_filtered["delivery_method_recode_7"].apply(
    lambda delivery_method: 1 if delivery_method == 2 else 0
)

df_filtered["successful_vbac"].describe()

In [ ]:
successful_vbac = df_filtered["successful_vbac"]

df_filtered = df_filtered.drop("successful_vbac", axis="columns")

In [ ]:
df_filtered.fillna(pd.NA)

df_filtered.describe()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_filtered, successful_vbac, train_size=0.8, random_state=14
)

In [ ]:
cat_features = [
    "birth_month",
    "birth_day_of_week",
    "birth_place",
    "facility_recode",
    "mothers_age_recode_14",
    "mothers_age_recode_9",
    "mothers_nativity",
    "mothers_residence_status",
    "mothers_race_recode_31",
    "mothers_race_recode_6",
    "mothers_race_recode_15",
    "mothers_hispanic_origin",
    "mothers_hispanic_origin_recode",
    "mothers_race_and_hispanic_origin",
    "paternity_acknowledged",
    "marital_status",
    "mothers_education",
    "fathers_age_recode_11",
    "fathers_race_recode_31",
    "fathers_race_recode_6",
    "fathers_race_recode_15",
    "fathers_hispanic_origin",
    "fathers_hispanic_origin_recode",
    "fathers_race_and_hispanic_origin",
    "fathers_education",
    "interval_since_last_live-birth_recode_11",
    "interval_since_last_other_pregnancy_recode",
    "interval_since_last_other_pregnancy_recode_11",
    "interval_since_last_pregnancy_recode",
    "interval_since_last_pregnancy_recode_11",
    "month_prenatal_care_began_recode",
    "number_of_prenatal_visits_recode",
    "WIC",
    "cigarettes_before_pregnancy_recode",
    "cigarettes_first_trimester_recode",
    "cigarettes_second_trimester_recode",
    "cigarettes_third_trimester_recode",
    "cigarette_recode",
    "BMI_recode",
    "pre_pregnancy_weight_recode",
    "delivery_weight_recode",
    "weight_gain_recode",
    "pre_pregnancy_diabetes",
    "gestational_diabetes",
    "pre_pregnancy_hypertension",
    "gestational_hypertension",
    "hypertension_eclampsia",
    "previous_preterm_birth",
    "infertility_treatment_used",
    "fertility_enhancing_drugs",
    "assistive_reproductive_technology",
    "previous_cesarean",
    "no_risk_factors_reported",
    "gonorrhea",
    "syphilis",
    "chlamydia",
    "hepatitis_b",
    "hepatitis_c",
    "no_infection_present",
    "successful_external_cephalic_version",
    "failed_external_cephalic_version",
    "induction_of_labor",
    "augmentation_of_labor",
    "steroids",
    "antibiotics",
    "chorioamnionitis",
    "anesthesia",
    "no_characteristics_of_labor_reported",
    "fetal_presentation_at_delivery",
    "final_route_and_method_of_delivery",
    "trial_of_labor_attempted_if_cesarean",
    "delivery_method_recode_7",
    "delivery_method_recode_3",
    "attendant_at_birth",
    "mother_transferred",
    "payment_source_for_delivery",
    "payment_recode",
    "sex_of_infant",
    "combined_gestation_recode_10",
    "combined_gestation_recode_3",
    "obstetric_estimate_recode_10",
    "obstetric_estimate_recode_3",
    "birth_weight_recode_12",
    "birth_weight_recode_4",
    "anencephaly",
    "spina_bifida",
    "cyanotic_congenital_heart_disease",
    "congenital_diaphragmatic_hernia",
    "omphalocele",
    "gastroschisis",
    "limb_reduction_defect",
    "cleft_lip",
    "cleft_palate",
    "down_syndrome",
    "suspected_chromosomal_disorder",
    "hypospadias",
    "no_congenital_abnormalities_reported",
]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

cat_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="most_frequent"),
    OneHotEncoder(drop="first", sparse_output=False),
)

In [ ]:
num_features = [
    "time_of_birth",
    "mothers_single_year_age",
    "fathers_combined_age",
    "prior_births_now_living",
    "prior_births_now_dead",
    "prior_other_terminations",
    "interval_since_last_live_birth",
    "number_of_prenatal_visits",
    "cigarettes_before_pregnancy",
    "cigarettes_first_trimester",
    "cigarettes_second_trimester",
    "cigarettes_third_trimester",
    "mothers_height_in_inches",
    "BMI",
    "weight_gain",
    "number_of_previous_cesarean",
    "combined_gestation_detail_in_weeks",
    "birth_weight_in_grams",
]

In [ ]:
from sklearn.preprocessing import StandardScaler

num_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="mean"), StandardScaler()
)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC

feature_select_pipeline = make_pipeline(SelectFromModel(LinearSVC(penalty="l1")))

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(
    [
        ("Categorical Pipeline", cat_pipeline, cat_features),
        ("Numerical Pipeline", num_pipeline, num_features),
        # ("Feature Selection Pipeline", feature_select_pipeline, X_train.columns.to_list())
    ]
)

In [ ]:
X_train_prepared = preprocessing.fit_transform(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_vbac = LogisticRegression(random_state=14, max_iter=1000)

log_vbac.fit(X_train_prepared, y_train)